In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [ ]:
from api.config.vehicle_profile_config import VehicleProfileConfig
from api.config.segmentation_config import SegmentationConfig
from api.config.output_config import OutputConfig

from moviasai.data.utils import load_raw_data
from moviasai.profiling.feature_extraction import get_extractor_by_prefix
from moviasai.profiling.profile import VersionedVehicleProfile

In [3]:
profile_cfg = VehicleProfileConfig.from_yaml('../config/vehicle_profile_config.yaml')
segmentation_cfg = SegmentationConfig.from_yaml('../config/segmentation_config.yaml')
output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')

def generate_profile(metric, output_cfg, profile_cfg, segmentation_cfg):
    df = load_raw_data(output_cfg.train_data_path(metric), target=metric)

    profile = VersionedVehicleProfile.from_config(
        config=profile_cfg,
        segmentation_config=segmentation_cfg,
        output_config=output_cfg,
        metric=metric,
        last=False,
        drop_duplicates=True,
    )

    profile.fit(df)

    profile.save(str(output_cfg.profile_dir(metric)))

    return profile

In [4]:
profile_km = generate_profile('km', output_cfg, profile_cfg, segmentation_cfg)
profile_h = generate_profile('h', output_cfg, profile_cfg, segmentation_cfg)

✓ Modelo ONNX carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.onnx
  Scaler (JSON): C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST_scaler.json
  Metadata: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST_metadata.json
  Nome: stage2_km
  Features: 5


AJUSTANDO PERFIS (KM)
Target: km_dia_clean
Veículos: 6718
Período: 2025-01-01 a 2025-10-10
Threads: 4
Batch size: 10
Extractors: 4
  • segmentation_features_km: 5 features
  • weekday_features_km: 49 features
  • month_phase_features_km: 15 features
  • monthly_cycle_features_km: 7 features
Classificador: stage2_km
  Features usadas: 5
  Classes: 3

📅 Total de semanas: 41
🔄 Processando em lotes paralelos...


Batch 5/5: 100%|██████████| 1/1 [00:11<00:00, 11.44s/it]


✓ Concluído em 160.54s
  - Versões: 207084
  - Duplicadas removidas: 22,676
  - Veículos: 6718
  - Distribuição de clusters:
      Cluster 1: 97,184 (46.9%)
      Cluster 0: 73,123 (35.3%)
      Cluster 2: 36,777 (17.8%)

✓ Salvos em: C:\Users\f0pi\git\apimovias\data\profiles\km
✓ Modelo ONNX carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST.onnx
  Scaler (JSON): C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST_scaler.json
  Metadata: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_h_BEST_metadata.json
  Nome: stage2_h
  Features: 5


AJUSTANDO PERFIS (H)
Target: h_dia_clean
Veículos: 1152
Período: 2025-01-01 a 2025-10-10
Threads: 4
Batch size: 10
Extractors: 4
  • segmentation_features_h: 5 features
  • weekday_features_h: 49 features
  • month_phase_features_h: 15 features
  • monthly_cycle_features_h: 7 features
Classificador: stage2_h
  Features usadas: 5
  Classes: 2

📅 Total de semanas: 41
🔄 Processando em lote

Batch 5/5: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


✓ Concluído em 24.76s
  - Versões: 31541
  - Duplicadas removidas: 7,166
  - Veículos: 1152
  - Distribuição de clusters:
      Cluster 1: 18,709 (59.3%)
      Cluster 0: 12,832 (40.7%)

✓ Salvos em: C:\Users\f0pi\git\apimovias\data\profiles\h


In [5]:
TARGET = 'km'
features = profile_cfg.features.km

df = load_raw_data(output_cfg.train_data_path(TARGET), target=TARGET)

pm = get_extractor_by_prefix('phase', features=features.phase, metric=TARGET)

df_ext = pm.extract(df)

In [6]:
seg_extractor = get_extractor_by_prefix('seg', features=features.seg, metric=TARGET)
wd_extractor = get_extractor_by_prefix('day', features=features.day, metric=TARGET)
pm_extractor = get_extractor_by_prefix('phase', features=features.phase, metric=TARGET)
mo_extractor = get_extractor_by_prefix('cycle', features=features.cycle, metric=TARGET)

In [7]:
for ex in [seg_extractor, wd_extractor, pm_extractor, mo_extractor]:
    print(ex.name)

    print('_feature_names: ', ex._feature_names)
    print('feature_names: ', ex.feature_names)


    print('selected_features: ', ex.selected_features)
    print('_selected_base: ', ex._selected_base)


    print()

segmentation_features_km
_feature_names:  ['seg_cv_gaps_km', 'seg_taxa_dias_ativos_km', 'seg_p75_km', 'seg_iqr_km']
feature_names:  ['seg_cv_gaps_km', 'seg_taxa_dias_ativos_km', 'seg_p75_km', 'seg_iqr_km']
selected_features:  ['cv_gaps', 'taxa_dias_ativos', 'p75', 'iqr']
_selected_base:  ['cv_gaps', 'taxa_dias_ativos', 'p75', 'iqr']

weekday_features_km
_feature_names:  ['day_1_mean_km', 'day_1_std_km', 'day_1_p25_km', 'day_1_p75_km', 'day_1_iqr_km', 'day_1_prob_active_km', 'day_1_cv_km', 'day_2_mean_km', 'day_2_std_km', 'day_2_p25_km', 'day_2_p75_km', 'day_2_iqr_km', 'day_2_prob_active_km', 'day_2_cv_km', 'day_3_mean_km', 'day_3_std_km', 'day_3_p25_km', 'day_3_p75_km', 'day_3_iqr_km', 'day_3_prob_active_km', 'day_3_cv_km', 'day_4_mean_km', 'day_4_std_km', 'day_4_p25_km', 'day_4_p75_km', 'day_4_iqr_km', 'day_4_prob_active_km', 'day_4_cv_km', 'day_5_mean_km', 'day_5_std_km', 'day_5_p25_km', 'day_5_p75_km', 'day_5_iqr_km', 'day_5_prob_active_km', 'day_5_cv_km', 'day_6_mean_km', 'day_6_st

In [8]:
import polars as pl

veiculos_test = (
    df.select("veiculo_id")
    .unique()
    .sample(n=2, seed=42)
    .to_series()
    .to_list()
)

df_test = df.filter(pl.col("veiculo_id").is_in(veiculos_test))
df_test


veiculo_id,data,km_dia_clean
i64,date,f64
14883,2025-01-01,0.0
14883,2025-01-02,0.0
14883,2025-01-03,176.6
14883,2025-01-04,1.6
14883,2025-01-05,0.0
…,…,…
23465,2025-10-06,26.3
23465,2025-10-07,15.1
23465,2025-10-08,28.9


In [9]:
df_res = wd_extractor.extract(df_test)
df_res

,veiculo_id,day_1_mean_km,day_1_std_km,day_1_p25_km,day_1_p75_km,day_1_iqr_km,day_1_prob_active_km,day_1_cv_km,day_2_mean_km,day_2_std_km,...,day_6_iqr_km,day_6_prob_active_km,day_6_cv_km,day_7_mean_km,day_7_std_km,day_7_p25_km,day_7_p75_km,day_7_iqr_km,day_7_prob_active_km,day_7_cv_km
0,14883,234.377500,209.920299,11.3,410.3,399.0,0.925000,0.895650,353.480000,260.402894,...,355.2,0.875,1.112068,150.617500,180.922723,0.0,376.0,376.0,0.67500,1.201207
1,23465,34.833333,19.488806,23.1,45.5,22.4,0.969697,0.559487,33.687879,19.481205,...,18.2,1.000,0.673990,31.923437,20.759852,18.2,41.4,23.2,0.96875,0.650301


In [10]:
for i in range(1, 8):
    print(f'Day {i} - cv_km')
    print(profile_km.df_versions[f'day_{i}_cv_km'].min())
    print(profile_km.df_versions[f'day_{i}_cv_km'].max())
    print()

Day 1 - cv_km
0.0
5.140437405874382

Day 2 - cv_km
0.0
5.318618756574472

Day 3 - cv_km
0.0
5.4157475611869526

Day 4 - cv_km
0.0
5.060274250729452

Day 5 - cv_km
0.0
5.001837211841232

Day 6 - cv_km
0.0
6.324555320336759

Day 7 - cv_km
0.0
6.32455532033676



In [11]:
vp_km = VersionedVehicleProfile.load(
    profile_dir=str(output_cfg.profile_dir('km')),
    classifier_path=str(output_cfg.classifier_path('km')),
)

✓ Modelo PKL carregado: C:\Users\f0pi\git\apimovias\models\classification\stage2\stage2_km_BEST.pkl
  Nome: stage2_km
  Features: 5
  Classes: 3

✓ Carregado: 6718 veículos, 207084 versões
  - Extractors: 4
    • segmentation_features_km: 5 features
    • weekday_features_km: 49 features
    • month_phase_features_km: 15 features
    • monthly_cycle_features_km: 7 features
  - Classificador: stage2_km


In [12]:
f = (vp_km.df_effective_period['veiculo_id'] == 21767) & (vp_km.df_effective_period['year'] == 2025) & (vp_km.df_effective_period['week'] == 1)
vp_km.df_effective_period[f]

,year,week,veiculo_id,dt_inicio,dt_fim
47,2025,1,21767,2024-12-30,2025-01-05


In [13]:
vp_km.all_feature_columns

['seg_gap_medio_km',
 'seg_cv_gaps_km',
 'seg_taxa_dias_ativos_km',
 'seg_p75_km',
 'seg_iqr_km',
 'day_1_mean_km',
 'day_1_std_km',
 'day_1_p25_km',
 'day_1_p75_km',
 'day_1_iqr_km',
 'day_1_prob_active_km',
 'day_1_cv_km',
 'day_2_mean_km',
 'day_2_std_km',
 'day_2_p25_km',
 'day_2_p75_km',
 'day_2_iqr_km',
 'day_2_prob_active_km',
 'day_2_cv_km',
 'day_3_mean_km',
 'day_3_std_km',
 'day_3_p25_km',
 'day_3_p75_km',
 'day_3_iqr_km',
 'day_3_prob_active_km',
 'day_3_cv_km',
 'day_4_mean_km',
 'day_4_std_km',
 'day_4_p25_km',
 'day_4_p75_km',
 'day_4_iqr_km',
 'day_4_prob_active_km',
 'day_4_cv_km',
 'day_5_mean_km',
 'day_5_std_km',
 'day_5_p25_km',
 'day_5_p75_km',
 'day_5_iqr_km',
 'day_5_prob_active_km',
 'day_5_cv_km',
 'day_6_mean_km',
 'day_6_std_km',
 'day_6_p25_km',
 'day_6_p75_km',
 'day_6_iqr_km',
 'day_6_prob_active_km',
 'day_6_cv_km',
 'day_7_mean_km',
 'day_7_std_km',
 'day_7_p25_km',
 'day_7_p75_km',
 'day_7_iqr_km',
 'day_7_prob_active_km',
 'day_7_cv_km',
 'phase_prob_

In [14]:
f = (vp_km.df_effective_period['veiculo_id'] == 41)
vp_km.df_effective_period[f]

,year,week,veiculo_id,dt_inicio,dt_fim
2549,2025,1,41,2024-12-30,2025-01-05
6705,2025,2,41,2024-12-30,2025-01-12
10997,2025,4,41,2024-12-30,2025-01-26
15496,2025,3,41,2024-12-30,2025-01-19
20184,2025,5,41,2024-12-30,2025-02-02
24643,2025,6,41,2024-12-30,2025-02-09
29257,2025,7,41,2024-12-30,2025-02-16
34117,2025,8,41,2024-12-30,2025-02-23
38542,2025,9,41,2024-12-30,2025-03-02
43250,2025,10,41,2024-12-30,2025-03-09


In [ ]:
vp_km.df_effective_period['dt_fim'].max()

Timestamp('2025-10-12 00:00:00')

In [19]:
vp_km.df_versions.isna().sum().sum()

np.int64(0)